In [0]:
# CELL 1: Parameters & Schema Definition
dbutils.widgets.text("wfp_file_path", "/Volumes/workspace/default/spi_data/samples/wfp_sample.csv")
wfp_file_path = dbutils.widgets.get("wfp_file_path")

from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from datetime import datetime
start_time = datetime.now()

# Redefined locally for session isolation
wfp_schema = StructType([
    StructField("adm0_id", IntegerType(), True), StructField("adm0_name", StringType(), True),
    StructField("adm1_id", IntegerType(), True), StructField("adm1_name", StringType(), True),
    StructField("mkt_id", IntegerType(), True), StructField("mkt_name", StringType(), True),
    StructField("cm_id", IntegerType(), True), StructField("cm_name", StringType(), True),
    StructField("cur_id", IntegerType(), True), StructField("cur_name", StringType(), True),
    StructField("pt_id", IntegerType(), True), StructField("pt_name", StringType(), True),
    StructField("um_id", IntegerType(), True), StructField("um_name", StringType(), True),
    StructField("mp_month", IntegerType(), True), StructField("mp_year", IntegerType(), True),
    StructField("mp_price", DoubleType(), True), StructField("mp_commoditysource", StringType(), True)
])

In [0]:
# CELL 2: Ingest and Write
# mergeSchema handles upstream source changes dynamically without failing the pipeline.
import uuid as _uuid

try:
    df_wfp = spark.read.format("csv").option("header", "true").schema(wfp_schema).load(wfp_file_path) \
        .withColumn("load_timestamp", current_timestamp())

    # Verification
    print("--- Raw WFP Preview ---")
    display(df_wfp.limit(3))

    df_wfp.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("default.bronze_wfp_raw")

    spark.sql(f"INSERT INTO default.pipeline_execution_logs VALUES ('{_uuid.uuid4()}', 'Raw-to-Bronze-WFP', 'wfp_sample.csv', '{start_time}', current_timestamp(), 'Success', {df_wfp.count()}, '')")
except Exception as e:
    spark.sql(f"INSERT INTO default.pipeline_execution_logs VALUES ('{_uuid.uuid4()}', 'Raw-to-Bronze-WFP', 'wfp_sample.csv', '{start_time}', current_timestamp(), 'Failure', 0, '{str(e)[0:200]}')")
    raise e

--- Raw WFP Preview ---


adm0_id,adm0_name,adm1_id,adm1_name,mkt_id,mkt_name,cm_id,cm_name,cur_id,cur_name,pt_id,pt_name,um_id,um_name,mp_month,mp_year,mp_price,mp_commoditysource,load_timestamp
null,Afghanistan,272,Badakhshan,266,Fayzabad,55,Bread - Retail,null,AFN,15,Retail,5,KG,1,2014,50.0,null,2026-10-07T18:52:48.109Z
null,Afghanistan,272,Badakhshan,266,Fayzabad,55,Bread - Retail,null,AFN,15,Retail,5,KG,2,2014,50.0,null,2026-10-07T18:52:48.109Z
null,Afghanistan,272,Badakhshan,266,Fayzabad,55,Bread - Retail,null,AFN,15,Retail,5,KG,3,2014,50.0,null,2026-10-07T18:52:48.109Z
